# Chapter 5: GPU Architecture and Hardware-Aware Decisions
### Companion notebook (High-Performance Machine Learning)

This notebook makes the chapter's claims measurable on a real GPU. It runs on a
**Colab T4** (Runtime → Change runtime type → T4 GPU) or any recent NVIDIA GPU. The book's numbers
were measured with Python 3.11, PyTorch 2.14.1 (CUDA 13.0), transformers 5.20.2 and sentence-transformers 6.2.0, measured on an NVIDIA A100 80GB (the book's reference), an H100 and a T4.

What you will measure here:

1. **Bandwidth vs compute**: a copy is bound by memory and a large matmul is bound by arithmetic.
2. **Coalescing**: contiguous access is far faster than strided access for the same byte count.
3. **Arithmetic intensity and the roofline**: where the RAG stages fall, computed from real shapes.
4. **Mixed precision (FP16)**: the honest result. FP16 speeds up the compute-bound prefill matmul and
   does little for memory-bound decode, exactly as Principle #14 predicts.
5. **Figure regeneration**: the code that produces every figure in the chapter.

> Solutions and extended versions of the exercises live at
> https://github.com/PacktPublishing/High-Performance-Machine-Learning

## 0. Setup

In [ ]:
import torch, time, math, gc
import numpy as np

assert torch.cuda.is_available(), 'Enable a GPU runtime (Colab: Runtime > Change runtime type > T4).'
dev = torch.device('cuda')
name = torch.cuda.get_device_name(0)
cap = torch.cuda.get_device_capability(0)
props = torch.cuda.get_device_properties(0)
print(f'GPU: {name}  (compute capability {cap[0]}.{cap[1]})')
print(f'SMs: {props.multi_processor_count}   global mem: {props.total_memory/1e9:.1f} GB')

# Turing (7.5) supports FP16 tensor cores but NOT BF16. We detect and report this.
IS_TURING = (cap == (7, 5))
BF16_OK = torch.cuda.is_bf16_supported()
print(f'Turing-class: {IS_TURING}   BF16 supported: {BF16_OK}')
if IS_TURING:
    print('=> On this card, mixed precision means FP16 (Section 5.4).')

In [ ]:
def cuda_time(fn, iters=50, warmup=10):
    """Median ms per call, measured with CUDA events."""
    for _ in range(warmup):
        fn()
    torch.cuda.synchronize()
    times = []
    for _ in range(iters):
        s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
        s.record(); fn(); e.record(); torch.cuda.synchronize()
        times.append(s.elapsed_time(e))
    return float(np.median(times))

## 1. Bandwidth vs compute

Two kernels. A **copy** does almost no arithmetic per byte, so it should be limited by memory
bandwidth. A large **square matmul** reuses each loaded element across a whole row/column, so it
should approach the compute peak. We measure the effective bandwidth of the first and the achieved
FLOP/s of the second, then compare to the card's ceilings.

In [ ]:
# --- memory-bound: big contiguous copy ---
N = 1 << 26  # 64M floats = 256 MB
x = torch.randn(N, device=dev)
y = torch.empty_like(x)
ms = cuda_time(lambda: y.copy_(x))
bytes_moved = 2 * x.numel() * x.element_size()  # read x + write y
gbs = bytes_moved / (ms * 1e-3) / 1e9
print(f'copy: {ms:.3f} ms   effective bandwidth ~ {gbs:.0f} GB/s')

# --- compute-bound: large FP16 matmul on tensor cores ---
M = 4096
a = torch.randn(M, M, device=dev, dtype=torch.float16)
b = torch.randn(M, M, device=dev, dtype=torch.float16)
ms_mm = cuda_time(lambda: torch.matmul(a, b))
flops = 2 * M**3  # multiply-add per output element
tflops = flops / (ms_mm * 1e-3) / 1e12
print(f'matmul {M}x{M} fp16: {ms_mm:.3f} ms   achieved ~ {tflops:.1f} TFLOP/s')

print('\nInterpretation: the copy sits near the memory ceiling; the matmul sits near the compute',
      'ceiling. Same chip, two different limits (Section 5.3).')

# --- self-check: copy is memory-bound, matmul is compute-bound ---
assert gbs > 50, f'effective bandwidth implausibly low: {gbs:.0f} GB/s'
assert tflops > 1, f'matmul throughput implausibly low: {tflops:.1f} TFLOP/s'
print(f'self-check passed: bandwidth ~{gbs:.0f} GB/s, matmul ~{tflops:.1f} TFLOP/s')


## 2. Coalescing: contiguous vs strided

Same number of elements gathered two ways. A contiguous gather lets the hardware merge the loads
into wide transactions; a strided gather scatters them. The byte counts are comparable, so any
timing gap is the access pattern, not the volume (Section 5.2, Figure 5.6).

In [ ]:
K = 1 << 22            # 4M elements gathered
base = torch.randn(1 << 26, device=dev)

contig_idx = torch.arange(K, device=dev)
stride = 64
strided_idx = (torch.arange(K, device=dev) * stride) % base.numel()

ms_c = cuda_time(lambda: base.index_select(0, contig_idx))
ms_s = cuda_time(lambda: base.index_select(0, strided_idx))
print(f'contiguous gather: {ms_c:.3f} ms')
print(f'strided gather   : {ms_s:.3f} ms   ({ms_s/ms_c:.1f}x slower for the same element count)')

# --- self-check: strided access is slower than contiguous ---
assert ms_s > ms_c, f'expected strided slower than contiguous, got contig={ms_c:.3f} strided={ms_s:.3f}'
print(f'self-check passed: strided gather is {ms_s/ms_c:.1f}x slower than contiguous')


## 3. Arithmetic intensity and the roofline

Compute the ridge point from the card's specs, then place the three RAG stages by their arithmetic
intensity. We use representative shapes for a ~2B-parameter generator; substitute your own model's
numbers to re-place the points.

In [ ]:
# Card ceilings (T4 reference values; adjust for your GPU).
PEAK_TFLOPS = 65.0      # FP16 tensor-core dense
PEAK_GBs    = 320.0
ridge = (PEAK_TFLOPS * 1e12) / (PEAK_GBs * 1e9)
print(f'ridge point ~ {ridge:.0f} FLOP/byte')

def intensity(flops, bytes_moved):
    return flops / bytes_moved

# ~2B params, fp16 weights = 2 bytes each.
P = 2.0e9
weight_bytes = P * 2

# decode: 1 token. ~2 FLOP per weight (one MAC), reads all weights once.
decode_flops = 2 * P * 1
decode_AI = intensity(decode_flops, weight_bytes)

# prefill: T tokens reuse the same weight reads.
T = 512
prefill_flops = 2 * P * T
prefill_AI = intensity(prefill_flops, weight_bytes)

# encoder: short batch-1 sequence, modest reuse.
Tenc = 32
enc_AI = intensity(2 * P * Tenc, weight_bytes)

for stg, ai in [('decode', decode_AI), ('encoder', enc_AI), ('prefill', prefill_AI)]:
    side = 'memory-bound' if ai < ridge else 'compute-bound'
    print(f'{stg:8s} AI ~ {ai:7.1f} FLOP/byte  -> {side}')

print('\nDecode is two orders of magnitude below the ridge: deeply memory-bound. Prefill is above it.')

# --- self-check: decode sits below the ridge, prefill above it ---
assert decode_AI < ridge < prefill_AI, \
    f'roofline ordering wrong: decode={decode_AI:.1f} ridge={ridge:.0f} prefill={prefill_AI:.1f}'
print(f'self-check passed: decode AI {decode_AI:.1f} < ridge {ridge:.0f} < prefill AI {prefill_AI:.1f}')


## 4. Mixed precision (FP16): the honest result

Principle #14 says Tensor Cores pay only where arithmetic is the bottleneck. We test that directly by
timing a **prefill-shaped** matmul (compute-bound) and a **decode-shaped** matvec (memory-bound) in
FP32 vs FP16.

- Prefill-shaped: a large matrix times a wide activation block (many tokens). FP16 should win clearly.
- Decode-shaped: the same weight matrix times a single-token vector. FP16 barely helps the math,
  because the kernel is waiting on weight reads; the only real gain there is reading half the bytes.

In [ ]:
H = 4096                 # hidden size proxy
FFN = 4 * H
Ttok = 512               # prefill block

def bench_linear(dtype, tokens):
    W = torch.randn(FFN, H, device=dev, dtype=dtype)
    Xb = torch.randn(H, tokens, device=dev, dtype=dtype)
    return cuda_time(lambda: torch.matmul(W, Xb))

# prefill-shaped (compute-bound)
p32 = bench_linear(torch.float32, Ttok)
p16 = bench_linear(torch.float16, Ttok)
print(f'prefill-shaped matmul  fp32 {p32:.3f} ms | fp16 {p16:.3f} ms | speedup {p32/p16:.2f}x')

# decode-shaped (memory-bound: 1 token)
d32 = bench_linear(torch.float32, 1)
d16 = bench_linear(torch.float16, 1)
print(f'decode-shaped  matvec  fp32 {d32:.3f} ms | fp16 {d16:.3f} ms | speedup {d32/d16:.2f}x')

print('\nExpect a solid speedup on the prefill-shaped matmul and a much smaller one on the',
      'decode-shaped matvec. The decode gain that does appear comes from moving fewer bytes',
      '(fp16 weights), not from faster tensor-core math, the memory-traffic lever of Chapter 9.')

# --- self-check: fp16 helps the compute-bound prefill more than the memory-bound decode ---
_prefill_speedup = p32 / p16
_decode_speedup = d32 / d16
assert _prefill_speedup > _decode_speedup, \
    f'expected prefill to gain more from fp16, got prefill={_prefill_speedup:.2f}x decode={_decode_speedup:.2f}x'
print(f'self-check passed: fp16 prefill speedup {_prefill_speedup:.2f}x > decode speedup {_decode_speedup:.2f}x')


### 4b. Optional: measure it on the real RAG models

This cell loads the actual case-study models and times prefill in FP32 vs FP16 and one decode step.
It downloads weights, so it is gated behind a flag. It reports the prefill delta honestly and shows
that decode is essentially unchanged, so we do **not** claim an end-to-end speedup from this lever.

In [ ]:
RUN_REAL_MODELS = False  # set True to download Gemma-2B-IT and time it

if RUN_REAL_MODELS:
    from transformers import AutoModelForCausalLM, AutoTokenizer
    mid = 'google/gemma-2b-it'
    tok = AutoTokenizer.from_pretrained(mid)
    prompt = 'Summarize the role of memory bandwidth in transformer inference.' * 8
    ids = tok(prompt, return_tensors='pt').input_ids.to(dev)

    def time_prefill(dtype):
        m = AutoModelForCausalLM.from_pretrained(mid, torch_dtype=dtype).to(dev).eval()
        with torch.no_grad():
            t = cuda_time(lambda: m(ids).logits, iters=10, warmup=3)
        del m; gc.collect(); torch.cuda.empty_cache()
        return t

    pf32 = time_prefill(torch.float32)
    pf16 = time_prefill(torch.float16)
    print(f'real prefill: fp32 {pf32:.1f} ms | fp16 {pf16:.1f} ms | speedup {pf32/pf16:.2f}x')
    print('Decode is dominated by per-token weight reads; halving weight bytes (fp16) helps via',
          'memory traffic, not tensor-core compute. See Chapter 9 for the accuracy-aware version.')
else:
    print('Set RUN_REAL_MODELS = True to run the full model measurement.')

## 5. Regenerate the chapter figures

The full figure generator is included so every figure in the chapter is reproducible. It writes the
grayscale (editorial) and terracotta (archive) PNGs and SVGs. Paste the contents of `make_figs.py`
from the solutions repo here, or run the trimmed roofline reproduction below as a check.

In [ ]:
import matplotlib.pyplot as plt

bw = PEAK_GBs * 1e9; peak = PEAK_TFLOPS * 1e12
AI = np.logspace(-1, 3.2, 400)
perf = np.minimum(AI * bw, peak) / 1e12
plt.figure(figsize=(7, 4.2))
plt.plot(AI, perf, lw=2.4, color='#404040')
plt.axvline(ridge, ls='--', color='#999999')
for lab, ai in [('decode', decode_AI), ('encoder', enc_AI), ('prefill', prefill_AI)]:
    y = min(ai * bw, peak) / 1e12
    plt.scatter([ai], [y], s=90, zorder=5)
    plt.annotate(f'{lab}\nAI~{ai:.0f}', (ai, y), textcoords='offset points', xytext=(6, -22))
plt.xscale('log'); plt.yscale('log')
plt.xlabel('arithmetic intensity (FLOP/byte)'); plt.ylabel('TFLOP/s')
plt.title('Roofline with the RAG stages (your hardware)')
plt.tight_layout(); plt.show()

print('Full multi-figure generator: https://github.com/PacktPublishing/High-Performance-Machine-Learning')

## 6. Exercises

These build on the case study from earlier chapters. Reference solutions are in the repo.

**5.1** Change `N` in Section 1 across `2**20 ... 2**28` and plot effective copy bandwidth vs size.
At what size does bandwidth plateau, and what does that say about cache effects?

**5.2** In Section 2, sweep `stride` over `1, 2, 8, 32, 128`. Plot slowdown vs stride and explain
the curve in terms of transaction width.

**5.3** Using your own model's parameter count and a prefill length you choose, recompute the three
arithmetic intensities and decide, before timing, which stages FP16 can speed up. Then verify with
Section 4.

**5.4** Vary `Ttok` in Section 4 from `1` to `1024`. Find the token count at which the prefill-shaped
matmul crosses from memory-bound to compute-bound on your card, and relate it to the ridge point.

**5.5** (Stretch) Set `RUN_REAL_MODELS = True`, then add a cell that times 32 decode steps in FP32
and FP16. Confirm the per-step time is set by weight reads and report whether FP16 weights change it,
and by how much. Tie your result back to Principle #14.